## 02 Silver Layer: Cleaning and Standardization

This notebook reads the Bronze procurement tables and cleans them into Silver tables.


### Silver tables created
- `silver_vendors`
- `silver_purchase_orders`
- `silver_line_items`
- `silver_spend_taxonomy`

### Main cleaning done
- standardized date formats
- clean currency fields
- standardize any text values like status, vendor name, and payment terms
- remove invalid records
- handle nulls only when needed

### Dependancies
- 01_bronze_ingestion.ipynb


In [0]:
#imports
import logging
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

In [0]:
#setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

log = logging.getLogger("procurement_silver")


#catalog and schema constants used in bronze
CATALOG = "healthcare_procurement"
SCHEMA = "procurement_analytics"

#starting logging
log.info("silver notebook started. Catalog=%s | Schema=%s", CATALOG, SCHEMA)

15:51:38  INFO  silver notebook started. Catalog=healthcare_procurement | Schema=procurement_analytics


In [0]:
#reusing same schema and catalog in bronze
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

log.info("Using %s.%s", CATALOG, SCHEMA)

15:51:56  INFO  Using healthcare_procurement.procurement_analytics


In [0]:
#reusable function definitions
#write
def write_table(df, table_name: str, mode: str = "overwrite"):
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    row_count = df.count()
    df.write.format("delta").mode(mode).saveAsTable(full_name)
    log.info("Written -> %s (%d rows)", full_name, row_count)

#generating null report
def null_report(df, label: str):
    print(f"\nNull report -- {label}")
    display(
        df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in df.columns
        ])
    )

#parsing date with different formats
def parse_date_multi(column_name: str):
    return F.coalesce(
        F.to_date(F.try_to_timestamp(F.col(column_name), F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(F.col(column_name), F.lit("MM/dd/yyyy"))),
        F.to_date(F.try_to_timestamp(F.col(column_name), F.lit("dd-MM-yyyy")))
    )

In [0]:
#reading bronze tables using spark
vendors_raw = spark.table(f"{CATALOG}.{SCHEMA}.bronze_vendors")
po_raw = spark.table(f"{CATALOG}.{SCHEMA}.bronze_purchase_orders")
lines_raw = spark.table(f"{CATALOG}.{SCHEMA}.bronze_line_items")
taxonomy_raw = spark.table(f"{CATALOG}.{SCHEMA}.bronze_spend_taxonomy")

log.info("all Bronze tables are now loaded successfully")

15:54:01  INFO  all Bronze tables are now loaded successfully


In [0]:
#debuggers
display(vendors_raw.limit(10))
display(po_raw.limit(10))
display(lines_raw.limit(10))
display(taxonomy_raw.limit(10))

15:54:37  INFO  Received command c on object id p0


vendor_id,vendor_name,city,country,payment_terms,vendor_category
V001,Medline Industries,null,USA,NET30,MEDICAL SUPPLIES
V002,cardinal health,Dublin,USA,NET 60,Medical Supplies
V003,johnson & johnson,New Brunswick,USA,net30,Medical Equipment
V004,Baxter International,Deerfield,USA,Net45,Pharmacy
V005,McKesson Corporation,null,USA,Net 30,Medical Supplies
V006,Stryker Corporation,Kalamazoo,USA,net60,MEDICAL EQUIPMENT
V007,bd (becton dickinson),Franklin Lakes,USA,net30,Medical Supplies
V008,Siemens Healthineers,Erlangen,Germany,NET 60,Medical Equipment
V009,Philips Healthcare,Amsterdam,Netherlands,NET45,Medical Equipment
V010,3m health care,Maplewood,USA,net30,Medical Supplies


po_id,po_date,paid_date,vendor_id,department_id,total_amount,status
PO-0001,18-02-2024,04/21/2024,V017,Surgery,46931,aprvd
PO-0002,2024-09-19,null,V007,Cardiology,42528.37,In Review
PO-0003,2024-11-24,null,V014,Oncology,26809,approved
PO-0004,12-09-2024,2024-11-24,V015,Radiology,47207,Approved
PO-0005,04/23/2023,2023-06-04,V020,oncology,16619,approved
PO-0006,2023-02-26,04/30/2023,V016,Administration,43224,approved
PO-0007,2024-09-14,2024-10-10,V007,Emergency,"$25,056.85",APPROVED
PO-0008,2023-05-01,2023-05-16,null,Emergency,41153.88,aprvd
PO-0009,2024-01-15,2024-03-16,V008,ICU,32737.51,APPROVED
PO-0010,2023-08-12,null,V003,Surgery,"$45,401.34",PENDING


line_id,po_id,item_description,quantity,unit_price
LI-00899,PO-0253,NEEDLE HOLDER MAYO,154,754.77
LI-00900,PO-0253,Printer Toner HP,144,307.58
LI-00901,PO-0253,Printer Toner HP,2,$412.60
LI-00902,PO-0253,Metformin 850mg,93,305.35
LI-00903,PO-0254,Disinfectant Spray 500ml,123,130.27
LI-00904,PO-0254,Trash Bags 50L,49,531.84
LI-00905,PO-0254,CATHETER URINARY,2,514.53
LI-00906,PO-0255,forceps surgical,87,761.72
LI-00907,PO-0255,SURGICAL GLOVES LARGE,22,120.54
LI-00908,PO-0255,NEEDLE HOLDER MAYO,147,342.68


taxonomy_id,keyword,subcategory,category
TAX-001,surgical gloves,PPE,Medical Supplies
TAX-002,n95,PPE,Medical Supplies
TAX-003,Face Masks,PPE,Medical Supplies
TAX-004,Protective Gowns,PPE,Medical Supplies
TAX-005,isolation gowns,PPE,Medical Supplies
TAX-006,iv bags,null,Medical Supplies
TAX-007,Syringes,Consumables,Medical Supplies
TAX-008,gauze,Consumables,Medical Supplies
TAX-009,bandages,Consumables,Medical Supplies
TAX-010,catheters,Consumables,Medical Supplies


### Silver: `silver_vendors`

Cleaning done here:
- replace null `city` and `country`
- standardize `vendor_name`
- normalize `payment_terms`
- standardize `vendor_category`

In [0]:
silver_vendors = (
    vendors_raw
    .withColumn(
        "city",
        F.when(F.col("city").isNull() | (F.trim(F.col("city")) == ""), "Unknown")
         .otherwise(F.trim(F.col("city")))
    )
    .withColumn(
        "country",
        F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""), "Unknown")
         .otherwise(F.trim(F.col("country")))
    )
    .withColumn(
        "vendor_name",
        F.initcap(
            F.regexp_replace(
                F.regexp_replace(
                    F.lower(F.trim(F.col("vendor_name"))),
                    "&",
                    " and "
                ),
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "payment_terms_digits",
        F.regexp_replace(F.lower(F.trim(F.col("payment_terms"))), r"[^0-9]", "")
    )
    .withColumn(
        "payment_terms",
        F.when(F.col("payment_terms_digits") == "30", "Net 30")
         .when(F.col("payment_terms_digits") == "45", "Net 45")
         .when(F.col("payment_terms_digits") == "60", "Net 60")
         .otherwise("Unknown")
    )
    .withColumn(
        "vendor_category",
        F.initcap(F.trim(F.col("vendor_category")))
    )
    .drop("payment_terms_digits")
)

In [0]:
#vendor validation
display(silver_vendors.limit(20))
null_report(silver_vendors, "silver_vendors")
silver_vendors.groupBy("payment_terms").count().show()

15:56:20  INFO  Received command c on object id p0


vendor_id,vendor_name,city,country,payment_terms,vendor_category
V001,Medline Industries,Unknown,USA,Net 30,Medical Supplies
V002,Cardinal Health,Dublin,USA,Net 60,Medical Supplies
V003,Johnson And Johnson,New Brunswick,USA,Net 30,Medical Equipment
V004,Baxter International,Deerfield,USA,Net 45,Pharmacy
V005,Mckesson Corporation,Unknown,USA,Net 30,Medical Supplies
V006,Stryker Corporation,Kalamazoo,USA,Net 60,Medical Equipment
V007,Bd (becton Dickinson),Franklin Lakes,USA,Net 30,Medical Supplies
V008,Siemens Healthineers,Erlangen,Germany,Net 60,Medical Equipment
V009,Philips Healthcare,Amsterdam,Netherlands,Net 45,Medical Equipment
V010,3m Health Care,Maplewood,USA,Net 30,Medical Supplies



Null report -- silver_vendors


vendor_id,vendor_name,city,country,payment_terms,vendor_category
0,0,0,0,0,0


+-------------+-----+
|payment_terms|count|
+-------------+-----+
|       Net 30|   10|
|       Net 60|    6|
|       Net 45|    4|
+-------------+-----+



In [0]:
#saving new vendors
write_table(silver_vendors, "silver_vendors")

15:56:49  INFO  Written -> healthcare_procurement.procurement_analytics.silver_vendors (20 rows)


### Silver: `silver_purchase_orders`

Cleaning done here:
- parsed `po_date` and `paid_date` mixed formats
- clean `total_amount`
- standardize `status`
- standardize `department_id`
- remove rows with null `vendor_id` because Gold joins depend on it

In [0]:
silver_purchase_orders = (
    po_raw
    .withColumn("po_date", parse_date_multi("po_date"))
    .withColumn("paid_date", parse_date_multi("paid_date"))
    .withColumn(
        "total_amount",
        F.regexp_replace(F.trim(F.col("total_amount")), r"[\$,]", "").cast(DoubleType())
    )
    .withColumn(
        "department_id",
        F.initcap(F.trim(F.col("department_id")))
    )
    .withColumn(
        "status_raw",
        F.lower(F.trim(F.col("status")))
    )
    .withColumn(
        "status",
        F.when(F.col("status_raw").isin("approved", "aprvd"), "Approved")
         .when(F.col("status_raw").isin("pending", "pndng", "in review"), "Pending")
         .when(F.col("status_raw").isin("rejected", "reject"), "Rejected")
         .otherwise("Unknown")
    )
    .drop("status_raw")
    .filter(F.col("vendor_id").isNotNull())
)

In [0]:
#purchase order validation
display(silver_purchase_orders.limit(20))
null_report(silver_purchase_orders, "silver_purchase_orders")
silver_purchase_orders.groupBy("status").count().show()

po_id,po_date,paid_date,vendor_id,department_id,total_amount,status
PO-0001,2024-02-18,2024-04-21,V017,Surgery,46931.0,Approved
PO-0002,2024-09-19,null,V007,Cardiology,42528.37,Pending
PO-0003,2024-11-24,null,V014,Oncology,26809.0,Approved
PO-0004,2024-09-12,2024-11-24,V015,Radiology,47207.0,Approved
PO-0005,2023-04-23,2023-06-04,V020,Oncology,16619.0,Approved
PO-0006,2023-02-26,2023-04-30,V016,Administration,43224.0,Approved
PO-0007,2024-09-14,2024-10-10,V007,Emergency,25056.85,Approved
PO-0009,2024-01-15,2024-03-16,V008,Icu,32737.51,Approved
PO-0010,2023-08-12,null,V003,Surgery,45401.34,Pending
PO-0011,2024-12-02,null,V014,Administration,32979.0,Pending



Null report -- silver_purchase_orders


po_id,po_date,paid_date,vendor_id,department_id,total_amount,status
0,0,145,0,0,0,0


+--------+-----+
|  status|count|
+--------+-----+
|Approved|  340|
| Pending|   96|
|Rejected|   45|
+--------+-----+



In [0]:
#check date parsing
silver_purchase_orders.select(
    "po_id", "po_date", "paid_date", "total_amount", "status", "department_id"
).show(20, truncate=False)

+-------+----------+----------+------------+--------+--------------+
|po_id  |po_date   |paid_date |total_amount|status  |department_id |
+-------+----------+----------+------------+--------+--------------+
|PO-0001|2024-02-18|2024-04-21|46931.0     |Approved|Surgery       |
|PO-0002|2024-09-19|NULL      |42528.37    |Pending |Cardiology    |
|PO-0003|2024-11-24|NULL      |26809.0     |Approved|Oncology      |
|PO-0004|2024-09-12|2024-11-24|47207.0     |Approved|Radiology     |
|PO-0005|2023-04-23|2023-06-04|16619.0     |Approved|Oncology      |
|PO-0006|2023-02-26|2023-04-30|43224.0     |Approved|Administration|
|PO-0007|2024-09-14|2024-10-10|25056.85    |Approved|Emergency     |
|PO-0009|2024-01-15|2024-03-16|32737.51    |Approved|Icu           |
|PO-0010|2023-08-12|NULL      |45401.34    |Pending |Surgery       |
|PO-0011|2024-12-02|NULL      |32979.0     |Pending |Administration|
|PO-0012|2024-01-24|2024-03-07|34778.0     |Approved|Facilities    |
|PO-0013|2024-11-11|NULL      |328

In [0]:
#save the new purchase orders
write_table(silver_purchase_orders, "silver_purchase_orders")

16:09:49  INFO  Written -> healthcare_procurement.procurement_analytics.silver_purchase_orders (481 rows)


### Silver: `silver_line_items`

Cleaning done here:
- removed rows with `quantity < or = 0`
- removed rows with null `item_description`
- cleaned `unit_price`
- standardized `item_description` to help later taxonomy matching

In [0]:
silver_line_items = (
    lines_raw
    .filter(F.col("quantity") > 0)
    .filter(F.col("item_description").isNotNull())
    .withColumn(
        "item_description",
        F.lower(
            F.regexp_replace(
                F.trim(F.col("item_description")),
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "unit_price",
        F.regexp_replace(F.trim(F.col("unit_price")), r"[\$,]", "").cast(DoubleType())
    )
)

In [0]:
#line items validation
display(silver_line_items.limit(20))
null_report(silver_line_items, "silver_line_items")
silver_line_items.filter(F.col("quantity") <= 0).show()

line_id,po_id,item_description,quantity,unit_price
LI-00899,PO-0253,needle holder mayo,154,754.77
LI-00900,PO-0253,printer toner hp,144,307.58
LI-00901,PO-0253,printer toner hp,2,412.6
LI-00902,PO-0253,metformin 850mg,93,305.35
LI-00903,PO-0254,disinfectant spray 500ml,123,130.27
LI-00904,PO-0254,trash bags 50l,49,531.84
LI-00905,PO-0254,catheter urinary,2,514.53
LI-00906,PO-0255,forceps surgical,87,761.72
LI-00907,PO-0255,surgical gloves large,22,120.54
LI-00908,PO-0255,needle holder mayo,147,342.68



Null report -- silver_line_items


line_id,po_id,item_description,quantity,unit_price
0,0,0,0,0


+-------+-----+----------------+--------+----------+
|line_id|po_id|item_description|quantity|unit_price|
+-------+-----+----------------+--------+----------+
+-------+-----+----------------+--------+----------+



In [0]:
#saving new line items
write_table(silver_line_items, "silver_line_items")

16:12:33  INFO  Written -> healthcare_procurement.procurement_analytics.silver_line_items (1657 rows)


### Silver: `silver_spend_taxonomy`

Cleaning done here:
- lowercased and trimed `keyword`
- standardized `subcategory` and `category`
- removed rows with null classification fields
- removed duplicate keywords

In [0]:
taxonomy_dedup_window = Window.partitionBy("keyword").orderBy("taxonomy_id")

silver_spend_taxonomy = (
    taxonomy_raw
    .withColumn("keyword", F.lower(F.trim(F.col("keyword"))))
    .withColumn("subcategory", F.initcap(F.trim(F.col("subcategory"))))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .dropna(subset=["keyword", "subcategory", "category"])
    .withColumn("_rn", F.row_number().over(taxonomy_dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn(
        "subcategory",
        #realized after validation changes (caps)
        F.when(F.col("subcategory") == "Ppe", "PPE")
         .otherwise(F.col("subcategory"))
    )
)

In [0]:
#taxonomy validation
display(silver_spend_taxonomy.limit(20))
null_report(silver_spend_taxonomy, "silver_spend_taxonomy")

silver_spend_taxonomy.groupBy("keyword").count() \
    .filter(F.col("count") > 1) \
    .show()

taxonomy_id,keyword,subcategory,category
TAX-023,amoxicillin,Medications,Pharmacy
TAX-028,antibiotics,Medications,Pharmacy
TAX-039,ballpoint pens,Stationery,Administrative
TAX-009,bandages,Consumables,Medical Supplies
TAX-011,catheter,Consumables,Medical Supplies
TAX-010,catheters,Consumables,Medical Supplies
TAX-035,cleaning cloths,Cleaning Supplies,Facilities
TAX-013,ct scanner,Imaging,Medical Equipment
TAX-029,disinfectant,Cleaning Supplies,Facilities
TAX-003,face masks,PPE,Medical Supplies



Null report -- silver_spend_taxonomy


taxonomy_id,keyword,subcategory,category
0,0,0,0


+-------+-----+
|keyword|count|
+-------+-----+
+-------+-----+



In [0]:
#saving new taxonomy 
write_table(silver_spend_taxonomy, "silver_spend_taxonomy")

16:15:00  INFO  Written -> healthcare_procurement.procurement_analytics.silver_spend_taxonomy (39 rows)


### Final validation

This section checks the final Silver tables after writing them.

In [0]:
summary = []

#for all print summary
for tbl in [
    "silver_vendors",
    "silver_purchase_orders",
    "silver_line_items",
    "silver_spend_taxonomy"
]:
    df = spark.table(f"{CATALOG}.{SCHEMA}.{tbl}")
    row_count = df.count()
    nulls = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
    null_summary = ", ".join(f"{c}={n}" for c, n in nulls.items() if n > 0)
    summary.append((tbl, row_count, null_summary if null_summary else "none"))

display(
    spark.createDataFrame(summary, ["table_name", "row_count", "null_columns"])
)

table_name,row_count,null_columns
silver_vendors,20,none
silver_purchase_orders,481,paid_date=145
silver_line_items,1657,none
silver_spend_taxonomy,39,none


In [0]:
#extra sanity checks for abnormal values
print("Distinct statuses:")
spark.table(f"{CATALOG}.{SCHEMA}.silver_purchase_orders").select("status").distinct().show()

print("Distinct payment terms:")
spark.table(f"{CATALOG}.{SCHEMA}.silver_vendors").select("payment_terms").distinct().show()

print("Any bad quantities left?")
spark.table(f"{CATALOG}.{SCHEMA}.silver_line_items").filter(F.col("quantity") <= 0).show()

print("Any duplicate taxonomy keywords left?")
spark.table(f"{CATALOG}.{SCHEMA}.silver_spend_taxonomy") \
    .groupBy("keyword").count() \
    .filter(F.col("count") > 1) \
    .show()

Distinct statuses:
+--------+
|  status|
+--------+
|Approved|
| Pending|
|Rejected|
+--------+

Distinct payment terms:
+-------------+
|payment_terms|
+-------------+
|       Net 30|
|       Net 60|
|       Net 45|
+-------------+

Any bad quantities left?
+-------+-----+----------------+--------+----------+
|line_id|po_id|item_description|quantity|unit_price|
+-------+-----+----------------+--------+----------+
+-------+-----+----------------+--------+----------+

Any duplicate taxonomy keywords left?
+-------+-----+
|keyword|count|
+-------+-----+
+-------+-----+

